In [31]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings("ignore")
import time

In [ ]:
# ccxt 라이브러리를 사용하여 업비트와 빗썸 거래소에서 XRP/KRW의 1분봉 데이터를 가져옵니다.
exchange_upb = ccxt.upbit()
exchange_bth = ccxt.bithumb()
data_upb = exchange_upb.fetch_ohlcv('XRP/KRW', '1m', limit=365)
data_bth = exchange_bth.fetch_ohlcv('XRP/KRW', '1m', limit=365)
df_upb = pd.DataFrame(data_upb, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df_bth = pd.DataFrame(data_bth, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df_upb['timestamp'] = pd.to_datetime(df_upb['timestamp'], unit='ms')
df_bth['timestamp'] = pd.to_datetime(df_bth['timestamp'], unit='ms')
#df_upb.set_index('timestamp', inplace=True)
#df_bth.set_index('timestamp', inplace=True)


In [11]:
df_upb
#df_bth


,timestamp,open,high,low,close,volume
0,2026-11-01 02:47:00,1683.0,1683.0,1681.0,1681.0,8153.949593
1,2026-11-01 02:48:00,1682.0,1683.0,1682.0,1683.0,11095.355009
2,2026-11-01 02:49:00,1683.0,1683.0,1681.0,1682.0,6617.071995
3,2026-11-01 02:50:00,1683.0,1683.0,1681.0,1681.0,84474.186420
4,2026-11-01 02:51:00,1682.0,1682.0,1681.0,1681.0,20881.895634
...,...,...,...,...,...,...
195,2026-11-01 06:02:00,1689.0,1689.0,1687.0,1688.0,15338.829779
196,2026-11-01 06:03:00,1689.0,1689.0,1687.0,1687.0,3776.804079
197,2026-11-01 06:04:00,1688.0,1688.0,1687.0,1687.0,8049.895335
198,2026-11-01 06:05:00,1688.0,1689.0,1687.0,1689.0,10331.684181


In [13]:
df_bth

,timestamp,open,high,low,close,volume
0,2026-11-01 02:47:00,1684.0,1685.0,1682.0,1683.0,1323.351526
1,2026-11-01 02:48:00,1684.0,1684.0,1683.0,1684.0,988.825203
2,2026-11-01 02:49:00,1684.0,1684.0,1683.0,1684.0,4465.933043
3,2026-11-01 02:50:00,1685.0,1685.0,1682.0,1682.0,1401.709386
4,2026-11-01 02:51:00,1683.0,1683.0,1682.0,1682.0,12083.680940
...,...,...,...,...,...,...
195,2026-11-01 06:02:00,1689.0,1689.0,1688.0,1689.0,153.457528
196,2026-11-01 06:03:00,1688.0,1689.0,1688.0,1689.0,313.221468
197,2026-11-01 06:04:00,1688.0,1689.0,1688.0,1688.0,14781.263830
198,2026-11-01 06:05:00,1689.0,1689.0,1688.0,1689.0,983.864593


In [ ]:
# vbt 라이브러리를 사용하여 업비트 거래소에서 XRP/KRW의 1분봉 데이터를 가져옵니다.
# => 44,640 개이어야 하나, len()이 그것에 모자람. 아울러 시간이 KST 기준이 아님
data_upb = vbt.CCXTData.download(
    exchange='upbit', 
    symbols='XRP/KRW', 
    timeframe='1m', 
    start='2026-10-01 00:00:00 KST', 
    end='2026-10-31 23:59:59 KST',
    retries=5,
    delay=100)

df = data_upb.get()
print(df.head())
print(df.tail())
print(len(df))

0it [00:00, ?it/s]

                             Open    High     Low   Close        Volume
Open time                                                              
2026-09-30 20:00:00+00:00  2027.0  2032.0  2026.0  2027.0  66684.780295
2026-09-30 20:01:00+00:00  2029.0  2030.0  2025.0  2026.0  97397.567819
2026-09-30 20:02:00+00:00  2028.0  2028.0  2026.0  2026.0  16089.552272
2026-09-30 20:03:00+00:00  2027.0  2028.0  2026.0  2028.0  14537.418609
2026-09-30 20:04:00+00:00  2028.0  2028.0  2025.0  2027.0  80450.772518
                             Open    High     Low   Close        Volume
Open time                                                              
2026-10-31 11:07:00+00:00  1682.0  1682.0  1681.0  1681.0   3108.268379
2026-10-31 11:08:00+00:00  1682.0  1682.0  1680.0  1680.0  34357.430563
2026-10-31 11:09:00+00:00  1681.0  1681.0  1680.0  1681.0  27389.719645
2026-10-31 11:10:00+00:00  1681.0  1681.0  1680.0  1680.0  10000.459639
2026-10-31 11:11:00+00:00  1681.0  1681.0  1680.0  1680.0   6267

In [50]:
# 
def fetch_korea_month_ohlcv(exchange='upbit', symbol='XRP/KRW', timeframe='1m', start_str='2026-10-01 00:00:00', end_str='2026-10-31 23:59:59'):
    exchange = ccxt.upbit({'enableRateLimit': True}) if exchange=='upbit' else ccxt.bithumb({'enableRateLimit': True})
    
    start_ts = int(pd.Timestamp(start_str, tz='Asia/Seoul').timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_str, tz='Asia/Seoul').timestamp() * 1000)
    
    timeframe_minutes = 1
    timeframe_ms = timeframe_minutes * 60 * 1000
    candle_count = ((end_ts - start_ts) // timeframe_ms) + 1
    
    all_candles = []
    to_ts = end_ts
    
    print(f"[{symbol}] 분봉 데이터 수집 시작 from {start_str} to {end_str}. candle_count: {candle_count}ea.")    

    while True:
        params = {'to': exchange.iso8601(to_ts)}
        try:
            candles = exchange.fetch_ohlcv(symbol=symbol, timeframe=timeframe, limit=200, params=params)
        except Exception as e:
            print(f"Error fetching data: {e}. Retrying in 2 seconds...")
            time.sleep(2)
            continue
        
        if not candles:
            break
        
        all_candles = candles + all_candles  # Prepend new candles to the list
        oldest_ts = candles[0][0]
        
        if oldest_ts <= start_ts:
            break
        
        to_ts = oldest_ts - 1  # Move to the next batch of candles
        time.sleep(0.2)  # Sleep to respect rate limits
        
    seen = set()
    deduped = []
    for c in all_candles:
        if c[0] not in seen and start_ts <= c[0] <= end_ts:
            seen.add(c[0])
            deduped.append(c)
            
    deduped.sort(key=lambda x: x[0])  # Sort by timestamp
    
    df = pd.DataFrame(deduped, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
    df.set_index('datetime', inplace=True)
    df.drop(columns=['timestamp'], inplace=True)
    return df


In [51]:
df_xrp_upb = fetch_korea_month_ohlcv(exchange='upbit', 
                                     symbol='XRP/KRW', 
                                     timeframe='1m',
                                     start_str='2026-10-01 00:00:00',
                                     end_str='2026-10-31 23:59:59')

print(f"Fetched {len(df_xrp_upb)} rows of data for XRP/KRW from Upbit.")
print(df_xrp_upb.head())
print(df_xrp_upb.tail())


[XRP/KRW] 분봉 데이터 수집 시작 from 2026-10-01 00:00:00 to 2026-10-31 23:59:59. candle_count: 44640ea.
Fetched 44336 rows of data for XRP/KRW from Upbit.
                             open    high     low   close        volume
datetime                                                               
2026-10-01 00:00:00+09:00  2035.0  2038.0  2034.0  2038.0  60728.189966
2026-10-01 00:01:00+09:00  2037.0  2038.0  2034.0  2038.0  18124.700376
2026-10-01 00:02:00+09:00  2038.0  2038.0  2036.0  2036.0  30253.493767
2026-10-01 00:03:00+09:00  2037.0  2038.0  2036.0  2038.0  43519.685875
2026-10-01 00:04:00+09:00  2038.0  2039.0  2037.0  2039.0  23692.105042
                             open    high     low   close        volume
datetime                                                               
2026-10-31 23:55:00+09:00  1683.0  1683.0  1681.0  1682.0   3753.306775
2026-10-31 23:56:00+09:00  1683.0  1683.0  1681.0  1683.0   5300.301500
2026-10-31 23:57:00+09:00  1683.0  1683.0  1681.0  1682.0  112

In [ ]:
# 02시부터 07시까지 데이터가 비어있음!!
df_xrp_upb = fetch_korea_month_ohlcv(exchange='upbit', 
                                     symbol='XRP/KRW', 
                                     timeframe='1m',
                                     start_str='2026-10-19 02:00:00',
                                     end_str='2026-10-19 07:59:59')

print(f"Fetched {len(df_xrp_upb)} rows of data for XRP/KRW from Upbit.")
print(df_xrp_upb.head())
print(df_xrp_upb.tail())


[XRP/KRW] 분봉 데이터 수집 시작 from 2026-10-19 02:00:00 to 2026-10-19 07:59:59...
Fetched 57 rows of data for XRP/KRW from Upbit.
                             open    high     low   close         volume
datetime                                                                
2026-10-19 02:00:00+09:00  1721.0  1721.0  1721.0  1721.0    2769.896344
2026-10-19 07:04:00+09:00  1722.0  1748.0  1722.0  1744.0  602520.230465
2026-10-19 07:05:00+09:00  1745.0  1749.0  1741.0  1746.0  256258.504050
2026-10-19 07:06:00+09:00  1745.0  1747.0  1744.0  1744.0   69839.754352
2026-10-19 07:07:00+09:00  1745.0  1745.0  1742.0  1745.0  179442.992992
                             open    high     low   close         volume
datetime                                                                
2026-10-19 07:55:00+09:00  1759.0  1759.0  1758.0  1759.0   12454.610261
2026-10-19 07:56:00+09:00  1759.0  1759.0  1758.0  1758.0   24824.500214
2026-10-19 07:57:00+09:00  1758.0  1759.0  1758.0  1758.0  135244.429225
20

In [ ]:
# vbt 라이브러리를 사용하여 이동평균 교차 전략을 구현하고 백테스트를 수행합니다. by Gemini
vbt_data = vbt.Data.from_data({'XRP/KRW': df_xrp_upb}, download_kwargs={})

price = vbt_data.get('close')
fast_ma = vbt.MA.run(price, window=10)
slow_ma = vbt.MA.run(price, window=30)

entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)
pf_1 = vbt.Portfolio.from_signals(
    close=price, 
    entries=entries, 
    exits=exits, 
    init_cash=100000, 
    fees=0.0005)
print(pf_1.stats())

Start                         2026-09-30 15:00:00+00:00
End                           2026-10-31 14:59:00+00:00
Period                                            44336
Start Value                                    100000.0
End Value                                  30537.959615
Total Return [%]                              -69.46204
Benchmark Return [%]                         -17.468106
Max Gross Exposure [%]                            100.0
Total Fees Paid                            52876.745894
Max Drawdown [%]                              69.643417
Max Drawdown Duration                           44251.0
Total Trades                                        899
Total Closed Trades                                 898
Total Open Trades                                     1
Open Trade PnL                                -15.26898
Win Rate [%]                                  18.374165
Best Trade [%]                                 2.961926
Worst Trade [%]                               -1

In [81]:
pf_1.plot().show()

In [ ]:
# ta 라이브러리를 사용하여 이동평균과 RSI 지표를 계산하고, 매수/매도 신호를 생성합니다. by Udemy
df_xrp_upb['ma10'] = ta.trend.SMAIndicator(close=df_xrp_upb['close'], window=10).sma_indicator()
df_xrp_upb['ma30'] = ta.trend.SMAIndicator(close=df_xrp_upb['close'], window=30).sma_indicator()

df_xrp_upb['rsi'] = ta.momentum.RSIIndicator(close=df_xrp_upb['close'], window=14).rsi()

df_xrp_upb['signal'] = np.where(df_xrp_upb['rsi'].between(0, 50), 1, -1)

conditions = [ df_xrp_upb['ma10'] < df_xrp_upb['ma30'], df_xrp_upb['ma30'] > df_xrp_upb['ma10'] ]
df_xrp_upb['signal2'] = np.select(conditions, [1, -1], default=0)

cond_go = [(df_xrp_upb['signal'] == 1) & (df_xrp_upb['signal2'] == 1), (df_xrp_upb['signal'] == -1) & (df_xrp_upb['signal2'] == -1)]
df_xrp_upb['go'] = np.select(cond_go, [1, -1], 0)
df_xrp_upb['go'].value_counts()

go
0    26557
1    17779
Name: count, dtype: int64

In [84]:
pf_2 = vbt.Portfolio.from_signals(
    close=df_xrp_upb['close'],
    entries=df_xrp_upb['go'] == 1,
    exits=df_xrp_upb['go'] == -1,
    init_cash=100000,
    tp_stop = 0.09,
    sl_stop = 0.02,
    short_entries = (df_xrp_upb['go'] == -1), # 
    short_exits = (df_xrp_upb['go'] == 1),
    fees=0.0005
)

pf_2.stats()

Start                         2026-10-01 00:00:00+09:00
End                           2026-10-31 23:59:00+09:00
Period                                            44336
Start Value                                    100000.0
End Value                                  80444.153499
Total Return [%]                             -19.555847
Benchmark Return [%]                         -17.468106
Max Gross Exposure [%]                            100.0
Total Fees Paid                              936.475112
Max Drawdown [%]                               23.66176
Max Drawdown Duration                           41715.0
Total Trades                                         11
Total Closed Trades                                  10
Total Open Trades                                     1
Open Trade PnL                               964.636358
Win Rate [%]                                        0.0
Best Trade [%]                                -2.102908
Worst Trade [%]                               -2

In [85]:
pf_2.plot().show()

In [86]:
pf_2.trades.records_readable

,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,0,48.851430,2026-10-01 01:32:00+09:00,2046.0,49.975012,2026-10-03 03:40:00+09:00,2005.0,48.973558,-2101.857185,-0.021029,Long,Closed,0
1,1,0,48.875733,2026-10-03 03:41:00+09:00,2002.0,48.924609,2026-10-07 20:38:00+09:00,1961.0,47.922657,-2100.752334,-0.021469,Long,Closed,1
2,2,0,48.801996,2026-10-07 20:39:00+09:00,1962.0,47.874758,2026-10-08 13:04:00+09:00,1921.0,46.874317,-2095.630902,-0.021887,Long,Closed,2
3,3,0,48.753218,2026-10-08 13:05:00+09:00,1921.0,46.827466,2026-10-09 00:20:00+09:00,1879.0,45.803648,-2140.266278,-0.022853,Long,Closed,3
4,4,0,48.652704,2026-10-09 00:21:00+09:00,1881.0,45.757868,2026-10-09 02:05:00+09:00,1843.0,44.833466,-1939.394070,-0.021192,Long,Closed,4
5,5,0,48.604075,2026-10-09 02:06:00+09:00,1843.0,44.788655,2026-10-13 09:08:00+09:00,1803.0,43.816574,-2032.768237,-0.022693,Long,Closed,5
6,6,0,48.287677,2026-10-13 09:09:00+09:00,1813.0,43.772779,2026-10-15 11:22:00+09:00,1770.0,42.734594,-2162.877480,-0.024706,Long,Closed,6
7,7,0,48.430948,2026-10-15 11:23:00+09:00,1763.0,42.691881,2026-10-18 22:35:00+09:00,1727.0,41.820124,-1828.026139,-0.021410,Long,Closed,7
8,8,0,48.298641,2026-10-18 22:36:00+09:00,1730.0,41.778325,2026-10-25 00:25:00+09:00,1695.0,40.933098,-1773.163860,-0.021221,Long,Closed,8
9,9,0,48.193501,2026-10-25 00:26:00+09:00,1697.0,40.892186,2026-10-30 09:18:00+09:00,1650.0,39.759638,-2345.746375,-0.028682,Long,Closed,9
